# Notebook 08: Natural Sample Application
Opx ML Thermobarometer
Author: [Your name]
Date: 2026-04-04

This notebook applies the trained models to natural peridotite samples and plots results on geotherms.

In [3]:
%matplotlib inline
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (ROOT, DATA_RAW, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
                    MODELS, FIGURES, RESULTS, LOGS, EXPETDB)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

## 8.1 Load natural sample data

In [4]:
NATURAL_CSV = DATA_NATURAL / 'lin_2023_ne_china_peridotites.csv'

if not NATURAL_CSV.exists():
    raise FileNotFoundError(
        f'Natural sample file not found: {NATURAL_CSV}\n'
        f'NB08 must not run on training data. Place a real natural sample CSV at:\n'
        f'  {DATA_NATURAL}\n'
        f'Download candidates: GEOROC, Viljoen et al. (2009), Lin et al. (2023), Qin et al. (2024).'
    )

df_natural = pd.read_csv(NATURAL_CSV)
df_natural = df_natural[df_natural[['SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO']].notna().all(axis=1)]

OXIDE_INFO = {
    'SiO2': (60.084, 1, 2), 'TiO2': (79.866, 1, 2), 'Al2O3': (101.961, 2, 3),
    'Cr2O3': (151.990, 2, 3), 'FeO_total': (71.844, 1, 1), 'MnO': (70.937, 1, 1),
    'MgO': (40.304, 1, 1), 'CaO': (56.077, 1, 1), 'Na2O': (61.979, 2, 1),
}

for ox, (mw, ncat, nox) in OXIDE_INFO.items():
    df_natural[f'{ox}_moles'] = df_natural[ox].fillna(0) / mw
    df_natural[f'{ox}_oxy'] = df_natural[f'{ox}_moles'] * nox

df_natural['total_oxy'] = sum(df_natural[f'{ox}_oxy'] for ox in OXIDE_INFO)
df_natural['norm_factor'] = 6.0 / df_natural['total_oxy']

CATION_NAMES = {'SiO2': 'Si', 'TiO2': 'Ti', 'Al2O3': 'Al', 'Cr2O3': 'Cr',
                'FeO_total': 'Fe', 'MnO': 'Mn', 'MgO': 'Mg', 'CaO': 'Ca', 'Na2O': 'Na'}
for ox, (mw, ncat, nox) in OXIDE_INFO.items():
    df_natural[f'cat_{CATION_NAMES[ox]}'] = df_natural[f'{ox}_moles'] * ncat * df_natural['norm_factor']

df_natural['cation_sum'] = df_natural[[f'cat_{c}' for c in CATION_NAMES.values()]].sum(axis=1)
df_natural = df_natural[(df_natural['cation_sum'] >= 3.95) & (df_natural['cation_sum'] <= 4.05)]

df_natural['sum_MgFeCa'] = df_natural['cat_Mg'] + df_natural['cat_Fe'] + df_natural['cat_Ca']
df_natural['Wo'] = df_natural['cat_Ca'] / df_natural['sum_MgFeCa'] * 100
df_natural = df_natural[df_natural['Wo'] <= 5.0]

df_natural['Mg_num'] = df_natural['cat_Mg'] / (df_natural['cat_Mg'] + df_natural['cat_Fe'])
df_natural['Al_IV'] = np.clip(2.0 - df_natural['cat_Si'], 0, df_natural['cat_Al'])
df_natural['Al_VI'] = df_natural['cat_Al'] - df_natural['Al_IV']
df_natural['En_frac'] = df_natural['cat_Mg'] / df_natural['sum_MgFeCa']
df_natural['Fs_frac'] = df_natural['cat_Fe'] / df_natural['sum_MgFeCa']
df_natural['Wo_frac'] = df_natural['cat_Ca'] / df_natural['sum_MgFeCa']
df_natural['MgTs'] = df_natural['Al_IV']

print(f'Natural samples loaded: {len(df_natural)}')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa6 in position 17: invalid start byte

## 8.2 Run predictions

In [ ]:
model_T_opx = joblib.load(MODELS / 'model_RF_T_C_opx_only.joblib')
model_P_opx = joblib.load(MODELS / 'model_RF_P_kbar_opx_only.joblib')

OPX_FEAT = ['SiO2','Al2O3','FeO_total','MgO','CaO','Mg_num','Al_IV','Al_VI','En_frac','Fs_frac','Wo_frac','MgTs']
X_natural = df_natural[OPX_FEAT].fillna(0).values

T_pred = model_T_opx.predict(X_natural)
P_pred = model_P_opx.predict(X_natural)

df_natural['T_pred'] = T_pred
df_natural['P_pred'] = P_pred

print(f'Predictions: T range {T_pred.min():.0f}-{T_pred.max():.0f} C, P range {P_pred.min():.1f}-{P_pred.max():.1f} kbar')

## 8.3 Plot on geotherms (Fig 12)

In [ ]:
def hasterok_chapman_geotherm(q_s_mW, z_km_array):
    """
    Hasterok & Chapman (2011) 1D steady-state conductive geotherm.
    Four-layer model: upper crust, middle crust, lower crust, mantle lithosphere.
    """
    q_s = q_s_mW * 1e-3  # W/m^2
    T_s = 10.0  # surface T (C)
    g = 9.81

    # (z_top_km, z_bot_km, A W/m^3, k W/m/K, rho kg/m^3)
    layers = [
        (0.0,  16.0, 1.30e-6, 2.5, 2800.0),
        (16.0, 24.0, 0.40e-6, 2.5, 2800.0),
        (24.0, 40.0, 0.40e-6, 2.5, 2800.0),
        (40.0, 300.0, 0.02e-6, 3.0, 3300.0),
    ]

    T = np.zeros_like(z_km_array, dtype=float)
    P = np.zeros_like(z_km_array, dtype=float)

    T_top = T_s
    q_top = q_s
    P_top = 0.0

    for (z0, z1, A, k, rho) in layers:
        mask = (z_km_array >= z0) & (z_km_array <= z1)
        dz_m = (z_km_array[mask] - z0) * 1000.0
        T[mask] = T_top + (q_top * dz_m - 0.5 * A * dz_m**2) / k
        P[mask] = P_top + rho * g * dz_m * 1e-8

        dz_layer = (z1 - z0) * 1000.0
        T_top = T_top + (q_top * dz_layer - 0.5 * A * dz_layer**2) / k
        q_top = q_top - A * dz_layer
        P_top = P_top + rho * g * dz_layer * 1e-8
        if z0 >= z_km_array.max():
            break

    return T, P

z_km = np.linspace(0, 250, 500)
T_40, P_40 = hasterok_chapman_geotherm(40, z_km)
T_45, P_45 = hasterok_chapman_geotherm(45, z_km)

# Sanity check at ~100 km (index 200 for 250km/500pts)
idx_100 = np.argmin(np.abs(z_km - 100))
print(f'Geotherm check -- 40 mW/m2 at {z_km[idx_100]:.0f} km: T={T_40[idx_100]:.0f} C, P={P_40[idx_100]:.1f} kbar')

P_max_plot = max(df_natural['P_pred'].max() * 1.2, 15.0)
mask_40 = P_40 <= P_max_plot
mask_45 = P_45 <= P_max_plot

fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(df_natural['T_pred'], df_natural['P_pred'],
                c=df_natural['Mg_num'], cmap='viridis',
                s=50, alpha=0.8, edgecolors='black', linewidths=0.5, zorder=3)
plt.colorbar(sc, ax=ax, label='Mg#')

ax.plot(T_40[mask_40], P_40[mask_40], 'r--', lw=1.5,
        label='40 mW/m2 geotherm (Hasterok & Chapman 2011)')
ax.plot(T_45[mask_45], P_45[mask_45], 'b--', lw=1.5,
        label='45 mW/m2 geotherm (Hasterok & Chapman 2011)')

ax.invert_yaxis()
ax.set_xlabel('Temperature (C)')
ax.set_ylabel('Pressure (kbar)')
ax.set_title('Fig 12: Predicted P-T on Geotherms (Natural Samples)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / 'fig12_natural_samples_geotherm.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()
print('Saved fig12_natural_samples_geotherm.png')

df_natural[['T_pred', 'P_pred', 'Mg_num']].to_csv(RESULTS / 'nb08_natural_predictions.csv', index=False)
print('\nNotebook 08 complete.')